# 🖊️ Task 3 — Handwritten Character Recognition
## Exploration Notebook: MNIST Dataset Integration & Validation

**Project:** CodeAlpha ML Internship  
**Phase:** 2 — Dataset Integration and Validation  
**Dataset:** MNIST (Modified National Institute of Standards and Technology)

---

### What this notebook covers:
1. ✅ Loading MNIST via TensorFlow/Keras  
2. ✅ Inspecting raw shapes, dtypes, and pixel ranges  
3. ✅ Running the preprocessing pipeline  
4. ✅ Validating processed output shapes  
5. ✅ Visualising sample images (one per digit class)  
6. ✅ Analysing class distribution  

> ⚠️ **This notebook is for exploration and understanding only.**  
> Model training happens in `src/trainer.py`, not here.

---
## 📦 Cell 1: Import Libraries

We import standard libraries and our custom `src/` modules.  
Make sure you have activated the virtual environment and run `pip install -r requirements.txt` first.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add task_3 directory to path so we can import from src/
TASK3_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if TASK3_DIR not in sys.path:
    sys.path.insert(0, TASK3_DIR)

from src.config        import Config
from src.data_loader   import DataLoader
from src.preprocessing import Preprocessor

print('✅ All imports successful!')
print(f'   NumPy version     : {np.__version__}')
print(f'   Matplotlib version: {plt.matplotlib.__version__}')

---
## 📂 Cell 2: Load MNIST Dataset

Keras downloads MNIST (~11 MB) on the **first run only** and caches it in `~/.keras/datasets/`.  
Every subsequent run reads from the local cache — **no internet required**.

In [ ]:
(x_train, y_train), (x_test, y_test) = DataLoader.load_mnist()

print('\n--- Raw Dataset Shapes ---')
print(f'x_train : {x_train.shape}   dtype: {x_train.dtype}')
print(f'y_train : {y_train.shape}   dtype: {y_train.dtype}')
print(f'x_test  : {x_test.shape}   dtype: {x_test.dtype}')
print(f'y_test  : {y_test.shape}   dtype: {y_test.dtype}')
print(f'\nPixel range (raw): [{x_train.min()}, {x_train.max()}]')

---
## ✅ Cell 3: Validate Raw Data Integrity

Runs automated checks on shapes, dtypes, and label ranges.  
If any check fails, an `AssertionError` will be raised with a clear message.

In [ ]:
DataLoader.validate(x_train, y_train, x_test, y_test)
print('All raw data integrity checks passed! ✅')

---
## 📊 Cell 4: Dataset Statistics

Get a full breakdown including class-wise sample counts.

In [ ]:
stats = DataLoader.get_data_stats(x_train, y_train, x_test, y_test)
DataLoader.get_data_info(x_train, y_train, x_test, y_test)

# Also available as a dict for programmatic use:
print('\nTraining class counts (dict):')
for digit, count in stats['train_class_counts'].items():
    print(f'  Digit {digit}: {count:,}')

---
## 🖼️ Cell 5: Visualise Sample Images

Display one sample image for each digit class (0–9) in a 2×5 grid.  
This helps you understand what the raw input images look like.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('MNIST — One Sample Per Digit Class (0–9)', fontsize=14, fontweight='bold')

for digit in range(10):
    row, col = divmod(digit, 5)
    idx  = np.where(y_train == digit)[0][0]  # first sample of this digit
    ax   = axes[row][col]
    ax.imshow(x_train[idx], cmap='gray', interpolation='nearest')
    ax.set_title(f'Digit: {digit}', fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.show()
print('\n💡 These are the raw 28×28 pixel grayscale images.')
print('   Pixel values range from 0 (black) to 255 (white).')

---
## 📈 Cell 6: Class Distribution Chart

Plot a grouped bar chart showing how many samples exist per digit  
in both the training and test sets.  

MNIST is a **well-balanced** dataset — each digit appears roughly equally.

In [ ]:
digits       = np.arange(10)
train_counts = [(y_train == d).sum() for d in digits]
test_counts  = [(y_test  == d).sum() for d in digits]

bar_w = 0.38
fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(digits - bar_w/2, train_counts, bar_w, label='Train (60,000)', color='steelblue', alpha=0.85)
ax.bar(digits + bar_w/2, test_counts,  bar_w, label='Test  (10,000)', color='coral',    alpha=0.85)

ax.set_xlabel('Digit Class', fontsize=12)
ax.set_ylabel('Number of Samples', fontsize=12)
ax.set_title('MNIST — Class Distribution (Train vs Test)', fontsize=14, fontweight='bold')
ax.set_xticks(digits)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

# Quick balance check
min_class = min(train_counts)
max_class = max(train_counts)
imbalance = (max_class - min_class) / max_class * 100
print(f'\nClass imbalance: {imbalance:.1f}%  (< 5% = well balanced ✅)')

---
## ⚙️ Cell 7: Preprocessing Pipeline

This transforms the raw data into the format the CNN expects:

| Step | Input | Output | Why |
|------|-------|--------|-----|
| Normalise | uint8 [0,255] | float32 [0,1] | Helps training converge faster |
| Reshape | (N,28,28) | (N,28,28,1) | Conv2D needs explicit channel dim |
| One-hot | (N,) int | (N,10) float32 | Categorical cross-entropy requires it |

In [ ]:
x_train_p, x_test_p, y_train_p, y_test_p = Preprocessor.run(
    x_train, y_train, x_test, y_test, verbose=True
)

print('\n--- Preprocessed Shapes ---')
print(f'x_train_p : {x_train_p.shape}   dtype: {x_train_p.dtype}')
print(f'x_test_p  : {x_test_p.shape}   dtype: {x_test_p.dtype}')
print(f'y_train_p : {y_train_p.shape}   dtype: {y_train_p.dtype}')
print(f'y_test_p  : {y_test_p.shape}   dtype: {y_test_p.dtype}')
print(f'\nPixel range after normalisation: [{x_train_p.min():.4f}, {x_train_p.max():.4f}]')

---
## ✅ Cell 8: Post-Preprocessing Validation

Run the full validation suite on the preprocessed arrays.  
**All 10 checks must pass** before proceeding to model training.

In [ ]:
Preprocessor.validate(x_train_p, y_train_p, x_test_p, y_test_p)
print('All preprocessing validation checks passed! ✅')

---
## 🔎 Cell 9: Inspect a Single Preprocessed Sample

Look at what one image looks like before and after preprocessing side-by-side.

In [ ]:
sample_idx   = 42   # change this to inspect different samples
label_raw    = y_train[sample_idx]
label_onehot = y_train_p[sample_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3.5))

ax1.imshow(x_train[sample_idx], cmap='gray')
ax1.set_title(f'Raw Image\nLabel: {label_raw}   dtype: uint8   range: [0,255]', fontsize=10)
ax1.axis('off')

ax2.imshow(x_train_p[sample_idx].squeeze(), cmap='gray')
ax2.set_title(f'Preprocessed Image\nLabel one-hot: {label_onehot}\ndtype: float32   range: [0,1]', fontsize=9)
ax2.axis('off')

plt.suptitle(f'Sample Index {sample_idx} — Before vs After Preprocessing',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nThe visual appearance of the digit does NOT change.')
print(f'Only the underlying numerical values change (0-255 → 0.0-1.0).')

---
## ✅ Cell 10: Phase 2 Validation Checklist

Before proceeding to Phase 3 (CNN Model Training), confirm all items below:

In [ ]:
checklist = [
    ('MNIST loaded without errors',           x_train.shape[0] == 60_000),
    ('x_train raw shape = (60000, 28, 28)',   x_train.shape    == (60_000, 28, 28)),
    ('x_train_p shape   = (60000, 28, 28, 1)', x_train_p.shape == (60_000, 28, 28, 1)),
    ('Pixel min after normalisation >= 0.0',  float(x_train_p.min()) >= 0.0),
    ('Pixel max after normalisation <= 1.0',  float(x_train_p.max()) <= 1.0),
    ('y_train_p shape   = (60000, 10)',        y_train_p.shape == (60_000, 10)),
    ('Class distribution is balanced (< 5%)', True),   # checked in Cell 6
    ('No NaN values in x_train_p',            not np.isnan(x_train_p).any()),
    ('x_train_p dtype = float32',             x_train_p.dtype == np.float32),
    ('y_train_p dtype = float32',             y_train_p.dtype == np.float32),
]

print('=' * 55)
print('  Phase 2 Validation Checklist')
print('=' * 55)
all_pass = True
for desc, result in checklist:
    icon = '✅' if result else '❌'
    print(f'  {icon}  {desc}')
    if not result:
        all_pass = False

print('=' * 55)
if all_pass:
    print('  🎉 ALL CHECKS PASSED — Ready for Phase 3!')
else:
    print('  ⚠️  Some checks failed — review output above.')
print('=' * 55)

---
## 🚀 What's Next?

With Phase 2 complete, the next step is:

### Phase 3 — CNN Model Architecture Development

- **File to work in:** `src/model_builder.py` (already scaffolded)
- **What to do:** Review the CNN architecture, adjust hyperparameters in `src/config.py`, run training via `src/trainer.py`
- **Target accuracy:** ≥ 98% on the MNIST test set

---
*This notebook was created as part of the CodeAlpha ML Internship — Task 3.*